# xBD label schema and old parser diagnostic
Read-only diagnostic; no model or training.


In [ ]:
from pathlib import Path
import json
from collections import Counter

root = Path('/kaggle/input')
paths = sorted(root.rglob('*_post_disaster.json'))
if not paths:
    raise RuntimeError('No post-disaster labels found; verify xBD attachment')
counts = Counter()
samples = []
for path in paths:
    payload = json.loads(path.read_text(encoding='utf-8'))
    features = payload.get('features')
    if not isinstance(features, dict):
        raise RuntimeError(f'Unexpected features structure: {path}')
    counts['labels'] += 1
    for key in ('xy', 'lng_lat'):
        value = features.get(key)
        if isinstance(value, list):
            counts[f'labels_with_{key}'] += 1
            counts[f'{key}_features'] += len(value)
            for feature in value:
                if not isinstance(feature, dict):
                    raise RuntimeError(f'Non-object feature in {path}')
                if isinstance(feature.get('wkt'), str):
                    counts[f'{key}_wkt_features'] += 1
                if isinstance(feature.get('geometry'), dict):
                    counts[f'{key}_geometry_features'] += 1
    selected = next((features[k] for k in ('xy', 'lng_lat', 'features') if isinstance(features.get(k), list)), [])
    counts['old_parser_selected_features'] += len(selected)
    counts['old_parser_accepted_geometry'] += sum(isinstance(f, dict) and isinstance(f.get('geometry'), dict) and f['geometry'].get('type') in ('Polygon', 'MultiPolygon') for f in selected)
    if len(samples) < 3:
        samples.append({'name': path.name, 'feature_keys': sorted(features), 'xy_feature_keys': sorted(features['xy'][0]) if isinstance(features.get('xy'), list) and features['xy'] else [], 'xy_wkt_prefix': features['xy'][0].get('wkt', '')[:120] if isinstance(features.get('xy'), list) and features['xy'] else None, 'lng_lat_feature_keys': sorted(features['lng_lat'][0]) if isinstance(features.get('lng_lat'), list) and features['lng_lat'] else []})
report = {'status': 'PASS', 'purpose': 'diagnose xBD source label structure and old training parser compatibility', 'counts': dict(counts), 'samples': samples, 'old_parser_dropped_all_selected_features': bool(counts['old_parser_selected_features'] and not counts['old_parser_accepted_geometry'])}
out = Path('/kaggle/working/rv4_label_diagnostic.json')
out.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2, sort_keys=True))
